# FINAL — Task 3 Evaluation and Consistency Checks

Questo notebook è pensato come **punto finale di consegna** del progetto.

L'obiettivo non è addestrare nuovi modelli, ma raccogliere in modo consistente gli output prodotti dai notebook precedenti e costruire una valutazione finale pulita:

1. creazione di un dataset Task 3 con split espliciti `QCD train / QCD validation / QCD test / EJ signal`;
2. costruzione delle label necessarie per ROC, AUC, efficienze a mistag fissato e supervised upper bound;
3. caricamento degli anomaly score prodotti da AE/VAE, RealNVP, Diffusion, Transformer e GNN opzionale;
4. valutazione QCD vs Emerging Jets;
5. controlli di consistenza QCD-vs-QCD;
6. confronto con baseline supervisionata;
7. sensitivity vs variabili truth-level LLP, se disponibili;
8. salvataggio di tutte le tabelle e figure in `FINAL/outputs/task3_evaluation`.

Nota metodologica: il segnale EJ viene usato solo in Task 3 per la valutazione finale e per il supervised upper bound.  
La scelta degli iperparametri dei modelli unsupervised deve essere fatta nei notebook di training usando solo QCD train/validation.

## 1. Import e configurazione generale

In [ ]:
from pathlib import Path
import json
import ast
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA

try:
    import h5py
except Exception:
    h5py = None

SEED = 42
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

CANDIDATE_BASES = [
    Path(".."),
    Path("."),
    Path("FINAL"),
]

PROJECT_BASE = None
for base in CANDIDATE_BASES:
    if (base / "processed").exists() or (base / "outputs").exists():
        PROJECT_BASE = base
        break

if PROJECT_BASE is None:
    PROJECT_BASE = Path("..")

PROCESSED_DIR = PROJECT_BASE / "processed"
OUTPUTS_DIR = PROJECT_BASE / "outputs"
TASK3_DIR = OUTPUTS_DIR / "task3_evaluation"
PLOTS_DIR = TASK3_DIR / "plots"
TABLES_DIR = TASK3_DIR / "tables"
DATASET_DIR = TASK3_DIR / "dataset"

for d in [TASK3_DIR, PLOTS_DIR, TABLES_DIR, DATASET_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_BASE:", PROJECT_BASE.resolve())
print("PROCESSED_DIR:", PROCESSED_DIR.resolve())
print("OUTPUTS_DIR:", OUTPUTS_DIR.resolve())
print("TASK3_DIR:", TASK3_DIR.resolve())

## 2. Parametri del Task 3

In [ ]:
# Se True, ricrea il dataset Task 3 anche se esiste già.
REBUILD_TASK3_DATASET = True

# Split QCD: se il preprocessing contiene solo bkg_eval,
# lo divido in validation e test in modo deterministico.
VAL_FRACTION_FROM_EVAL = 0.50
TEST_FRACTION_FROM_EVAL = 0.50

# Se vuoi limitare la dimensione per test veloci, metti un intero.
# Per il run finale lascia None.
MAX_BKG_TRAIN = None
MAX_BKG_VAL = None
MAX_BKG_TEST = None
MAX_SIG = None

# Per il supervised upper bound.
SUPERVISED_MAX_TRAIN_PER_CLASS = 200_000
SUPERVISED_MAX_TEST_PER_CLASS = None

# Mistag rate usati nella tabella finale.
MISTAG_POINTS = [0.30, 0.10, 0.05, 0.01, 0.001]

# Bootstrap opzionale per incertezze statistiche.
# Può essere lento con molti score/modelli; per il primo run si può lasciare False.
DO_BOOTSTRAP_CI = False
N_BOOTSTRAP = 200
BOOTSTRAP_MAX_EVENTS_PER_CLASS = 100_000

# Truth-level sensitivity.
DO_TRUTH_SENSITIVITY = True
TRUTH_MISTAG_POINTS = [0.01, 0.001]
N_TRUTH_BINS = 6

# Se il preprocessing non contiene già truth-level summaries,
# puoi indicare qui il path al file HDF5 signal raw.
# Deve contenere il dataset/group "truth_dark_pions".
RAW_SIGNAL_H5_PATH = None

print("MISTAG_POINTS:", MISTAG_POINTS)
print("DO_BOOTSTRAP_CI:", DO_BOOTSTRAP_CI)
print("DO_TRUTH_SENSITIVITY:", DO_TRUTH_SENSITIVITY)

## 3. Utility di I/O e controlli anti-leakage

In [ ]:
FORBIDDEN_FEATURE_SUBSTRINGS = [
    "isDisplaced",
    "isTagged",
    "salt_pdisp",
    "displacedPtFraction",
    "flavour_label",
    "truthOriginLabel",
    "truthVertexIndex",
    "VSIVertexIndex",
    "truth_dark_pions",
    "truth_stable_non_geant",
    "eventNumber",
    "mcEventWeight",
    "averageInteractionsPerCrossing",
    "actualInteractionsPerCrossing",
    "nPrimaryVertices",
    "isRun3",
]


def safe_load_npz(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return np.load(path, allow_pickle=True)
    except Exception as e:
        print(f"[ERROR] non riesco a leggere NPZ {path}: {e}")
        return None


def safe_read_csv(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[ERROR] non riesco a leggere CSV {path}: {e}")
        return None


def first_existing_key(data, candidates, required=True):
    if data is None:
        if required:
            raise KeyError("data is None")
        return None

    for key in candidates:
        if key in data:
            return key

    if required:
        raise KeyError(
            "Nessuna chiave trovata tra: "
            + ", ".join(candidates)
            + "\nChiavi disponibili: "
            + ", ".join(list(data.keys()))
        )

    return None


def as_list_of_str(x):
    if x is None:
        return []
    arr = np.asarray(x)
    if arr.ndim == 0:
        return [str(arr.item())]
    return [str(v) for v in arr.tolist()]


def check_no_forbidden_features(feature_names, context="features"):
    feature_names = [str(x) for x in feature_names]
    bad = []

    for f in feature_names:
        for token in FORBIDDEN_FEATURE_SUBSTRINGS:
            if token.lower() in f.lower():
                bad.append(f)

    if bad:
        raise RuntimeError(
            f"[{context}] Feature potenzialmente leaky trovate: "
            + ", ".join(sorted(set(bad)))
        )

    print(f"[OK] anti-leakage check: {context}")


def finite_1d(x):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    return x[np.isfinite(x)]


def subsample_array(x, max_n, seed):
    if max_n is None or len(x) <= max_n:
        return x

    rr = np.random.default_rng(seed)
    idx = rr.choice(len(x), size=max_n, replace=False)
    return x[idx]


def take_indices(x, idx):
    if x is None:
        return None
    return np.asarray(x)[idx]


def limit_split(X, max_n, seed):
    if X is None:
        return None
    if max_n is None or len(X) <= max_n:
        return X
    rr = np.random.default_rng(seed)
    idx = rr.choice(len(X), size=max_n, replace=False)
    return X[idx]


def save_json(obj, path):
    path = Path(path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def ensure_2d_float(x, name):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2:
        raise ValueError(f"{name} deve essere 2D, shape trovata: {x.shape}")
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

## 4. Creazione dataset Task 3 con label e split espliciti

In [ ]:
AGG_PATH = PROCESSED_DIR / "repr1_aggregate.npz"
SEQ_PATH = PROCESSED_DIR / "repr2_sequences.npz"
TASK3_DATASET_PATH = DATASET_DIR / "task3_eval_dataset.npz"
TASK3_METADATA_PATH = DATASET_DIR / "task3_eval_dataset_metadata.json"

print("AGG_PATH:", AGG_PATH)
print("SEQ_PATH:", SEQ_PATH)
print("TASK3_DATASET_PATH:", TASK3_DATASET_PATH)


def get_agg_arrays(agg):
    bkg_train_key = first_existing_key(
        agg,
        ["bkg_train", "X_agg_bkg_train", "X_bkg_train", "X_train_bkg"],
    )

    bkg_eval_key = first_existing_key(
        agg,
        ["bkg_eval", "X_agg_bkg_eval", "X_bkg_eval", "X_eval_bkg"],
        required=False,
    )

    bkg_val_key = first_existing_key(
        agg,
        ["bkg_val", "X_agg_bkg_val", "X_bkg_val", "X_val_bkg"],
        required=False,
    )

    bkg_test_key = first_existing_key(
        agg,
        ["bkg_test", "X_agg_bkg_test", "X_bkg_test", "X_test_bkg"],
        required=False,
    )

    sig_key = first_existing_key(
        agg,
        ["sig", "X_agg_sig", "X_sig", "X_signal"],
    )

    feature_key = first_existing_key(
        agg,
        ["feature_names", "agg_feature_names", "aggregate_feature_names"],
        required=False,
    )

    X_train = ensure_2d_float(agg[bkg_train_key], bkg_train_key)
    X_sig = ensure_2d_float(agg[sig_key], sig_key)

    if bkg_val_key is not None and bkg_test_key is not None:
        X_val = ensure_2d_float(agg[bkg_val_key], bkg_val_key)
        X_test = ensure_2d_float(agg[bkg_test_key], bkg_test_key)
        split_source = "explicit_bkg_val_and_bkg_test"
    elif bkg_eval_key is not None:
        X_eval = ensure_2d_float(agg[bkg_eval_key], bkg_eval_key)
        idx = np.arange(len(X_eval))
        idx_val, idx_test = train_test_split(
            idx,
            test_size=TEST_FRACTION_FROM_EVAL,
            random_state=SEED,
            shuffle=True,
        )
        X_val = X_eval[idx_val]
        X_test = X_eval[idx_test]
        split_source = "split_from_bkg_eval"
    else:
        idx = np.arange(len(X_train))
        idx_train_new, idx_tmp = train_test_split(
            idx,
            test_size=0.30,
            random_state=SEED,
            shuffle=True,
        )
        idx_val, idx_test = train_test_split(
            idx_tmp,
            test_size=0.50,
            random_state=SEED,
            shuffle=True,
        )
        X_train_original = X_train
        X_train = X_train_original[idx_train_new]
        X_val = X_train_original[idx_val]
        X_test = X_train_original[idx_test]
        split_source = "split_from_bkg_train_fallback"

    if feature_key is not None:
        feature_names = as_list_of_str(agg[feature_key])
    else:
        feature_names = [f"agg_f{i}" for i in range(X_train.shape[1])]

    check_no_forbidden_features(feature_names, "aggregate features")

    return {
        "agg_X_bkg_train": X_train,
        "agg_X_bkg_val": X_val,
        "agg_X_bkg_test": X_test,
        "agg_X_sig": X_sig,
        "agg_feature_names": np.array(feature_names, dtype=object),
        "agg_split_source": split_source,
    }


def get_seq_arrays(seq):
    if seq is None:
        return {}

    x_train_key = first_existing_key(seq, ["X_bkg_train", "X_tracks_bkg_train", "tracks_bkg_train"], required=False)
    m_train_key = first_existing_key(seq, ["mask_bkg_train", "track_mask_bkg_train", "mask_train"], required=False)
    j_train_key = first_existing_key(seq, ["jet_bkg_train", "X_jets_bkg_train", "jets_bkg_train"], required=False)

    x_eval_key = first_existing_key(seq, ["X_bkg_eval", "X_tracks_bkg_eval", "tracks_bkg_eval"], required=False)
    m_eval_key = first_existing_key(seq, ["mask_bkg_eval", "track_mask_bkg_eval", "mask_eval"], required=False)
    j_eval_key = first_existing_key(seq, ["jet_bkg_eval", "X_jets_bkg_eval", "jets_bkg_eval"], required=False)

    x_val_key = first_existing_key(seq, ["X_bkg_val", "X_tracks_bkg_val", "tracks_bkg_val"], required=False)
    m_val_key = first_existing_key(seq, ["mask_bkg_val", "track_mask_bkg_val", "mask_val"], required=False)
    j_val_key = first_existing_key(seq, ["jet_bkg_val", "X_jets_bkg_val", "jets_bkg_val"], required=False)

    x_test_key = first_existing_key(seq, ["X_bkg_test", "X_tracks_bkg_test", "tracks_bkg_test"], required=False)
    m_test_key = first_existing_key(seq, ["mask_bkg_test", "track_mask_bkg_test", "mask_test"], required=False)
    j_test_key = first_existing_key(seq, ["jet_bkg_test", "X_jets_bkg_test", "jets_bkg_test"], required=False)

    x_sig_key = first_existing_key(seq, ["X_sig", "X_tracks_sig", "tracks_sig", "X_signal"], required=False)
    m_sig_key = first_existing_key(seq, ["mask_sig", "track_mask_sig", "mask_signal"], required=False)
    j_sig_key = first_existing_key(seq, ["jet_sig", "X_jets_sig", "jets_sig", "X_jets_signal"], required=False)

    track_names_key = first_existing_key(seq, ["track_feature_names", "tracks_feature_names"], required=False)
    jet_names_key = first_existing_key(seq, ["jet_feature_names", "jets_feature_names"], required=False)

    if x_train_key is None or m_train_key is None or x_sig_key is None or m_sig_key is None:
        print("[INFO] repr2_sequences non contiene tutte le chiavi necessarie; salto salvataggio sequenze Task 3.")
        return {}

    X_train = np.asarray(seq[x_train_key], dtype=np.float32)
    M_train = np.asarray(seq[m_train_key]).astype(bool)
    X_sig = np.asarray(seq[x_sig_key], dtype=np.float32)
    M_sig = np.asarray(seq[m_sig_key]).astype(bool)

    if j_train_key is not None:
        J_train = np.asarray(seq[j_train_key], dtype=np.float32)
    else:
        J_train = np.zeros((len(X_train), 0), dtype=np.float32)

    if j_sig_key is not None:
        J_sig = np.asarray(seq[j_sig_key], dtype=np.float32)
    else:
        J_sig = np.zeros((len(X_sig), J_train.shape[1]), dtype=np.float32)

    if x_val_key is not None and m_val_key is not None and x_test_key is not None and m_test_key is not None:
        X_val = np.asarray(seq[x_val_key], dtype=np.float32)
        M_val = np.asarray(seq[m_val_key]).astype(bool)
        X_test = np.asarray(seq[x_test_key], dtype=np.float32)
        M_test = np.asarray(seq[m_test_key]).astype(bool)

        J_val = np.asarray(seq[j_val_key], dtype=np.float32) if j_val_key is not None else np.zeros((len(X_val), J_train.shape[1]), dtype=np.float32)
        J_test = np.asarray(seq[j_test_key], dtype=np.float32) if j_test_key is not None else np.zeros((len(X_test), J_train.shape[1]), dtype=np.float32)

        split_source = "explicit_bkg_val_and_bkg_test"

    elif x_eval_key is not None and m_eval_key is not None:
        X_eval = np.asarray(seq[x_eval_key], dtype=np.float32)
        M_eval = np.asarray(seq[m_eval_key]).astype(bool)
        J_eval = np.asarray(seq[j_eval_key], dtype=np.float32) if j_eval_key is not None else np.zeros((len(X_eval), J_train.shape[1]), dtype=np.float32)

        idx = np.arange(len(X_eval))
        idx_val, idx_test = train_test_split(
            idx,
            test_size=TEST_FRACTION_FROM_EVAL,
            random_state=SEED,
            shuffle=True,
        )

        X_val, M_val, J_val = X_eval[idx_val], M_eval[idx_val], J_eval[idx_val]
        X_test, M_test, J_test = X_eval[idx_test], M_eval[idx_test], J_eval[idx_test]
        split_source = "split_from_bkg_eval"

    else:
        print("[INFO] repr2_sequences non contiene eval/test QCD; salto split sequenziale completo.")
        return {}

    if track_names_key is not None:
        track_feature_names = as_list_of_str(seq[track_names_key])
    else:
        track_feature_names = [f"track_f{i}" for i in range(X_train.shape[-1])]

    if jet_names_key is not None:
        jet_feature_names = as_list_of_str(seq[jet_names_key])
    else:
        jet_feature_names = [f"jet_f{i}" for i in range(J_train.shape[-1])]

    check_no_forbidden_features(track_feature_names, "track features")
    check_no_forbidden_features(jet_feature_names, "jet features")

    return {
        "seq_X_bkg_train": X_train,
        "seq_M_bkg_train": M_train,
        "seq_J_bkg_train": J_train,
        "seq_X_bkg_val": X_val,
        "seq_M_bkg_val": M_val,
        "seq_J_bkg_val": J_val,
        "seq_X_bkg_test": X_test,
        "seq_M_bkg_test": M_test,
        "seq_J_bkg_test": J_test,
        "seq_X_sig": X_sig,
        "seq_M_sig": M_sig,
        "seq_J_sig": J_sig,
        "track_feature_names": np.array(track_feature_names, dtype=object),
        "jet_feature_names": np.array(jet_feature_names, dtype=object),
        "seq_split_source": split_source,
    }


def create_task3_dataset():
    agg = safe_load_npz(AGG_PATH)
    if agg is None:
        raise FileNotFoundError(f"Non trovo {AGG_PATH}. Prima devi eseguire FINAL_preprocessing.ipynb.")

    seq = safe_load_npz(SEQ_PATH)

    out = {}
    out.update(get_agg_arrays(agg))
    out.update(get_seq_arrays(seq))

    # Limiti opzionali per run rapidi.
    out["agg_X_bkg_train"] = limit_split(out["agg_X_bkg_train"], MAX_BKG_TRAIN, SEED)
    out["agg_X_bkg_val"] = limit_split(out["agg_X_bkg_val"], MAX_BKG_VAL, SEED + 1)
    out["agg_X_bkg_test"] = limit_split(out["agg_X_bkg_test"], MAX_BKG_TEST, SEED + 2)
    out["agg_X_sig"] = limit_split(out["agg_X_sig"], MAX_SIG, SEED + 3)

    # Label per valutazione finale aggregate.
    out["agg_y_bkg_test_sig"] = np.concatenate([
        np.zeros(len(out["agg_X_bkg_test"]), dtype=np.int64),
        np.ones(len(out["agg_X_sig"]), dtype=np.int64),
    ])

    out["agg_sample_bkg_test_sig"] = np.array(
        ["QCD"] * len(out["agg_X_bkg_test"]) + ["EJ"] * len(out["agg_X_sig"]),
        dtype=object,
    )

    metadata = {
        "seed": SEED,
        "agg_split_source": str(out.get("agg_split_source", "")),
        "seq_split_source": str(out.get("seq_split_source", "")),
        "n_agg_bkg_train": int(len(out["agg_X_bkg_train"])),
        "n_agg_bkg_val": int(len(out["agg_X_bkg_val"])),
        "n_agg_bkg_test": int(len(out["agg_X_bkg_test"])),
        "n_agg_sig": int(len(out["agg_X_sig"])),
        "n_agg_features": int(out["agg_X_bkg_train"].shape[1]),
        "notes": [
            "QCD train is used for unsupervised training in model notebooks.",
            "QCD validation is used for early stopping, hyperparameter tuning, and score calibration.",
            "QCD test and EJ signal are used for final Task 3 evaluation.",
            "EJ signal is not used for unsupervised model training.",
        ],
    }

    np.savez_compressed(TASK3_DATASET_PATH, **out)
    save_json(metadata, TASK3_METADATA_PATH)

    return out, metadata


if TASK3_DATASET_PATH.exists() and not REBUILD_TASK3_DATASET:
    task3_data = np.load(TASK3_DATASET_PATH, allow_pickle=True)
    with TASK3_METADATA_PATH.open("r", encoding="utf-8") as f:
        task3_metadata = json.load(f)
else:
    task3_data_dict, task3_metadata = create_task3_dataset()
    task3_data = np.load(TASK3_DATASET_PATH, allow_pickle=True)

print("Task 3 dataset creato/caricato:")
for k in task3_data.keys():
    arr = task3_data[k]
    print(f"  {k:30s} shape={arr.shape} dtype={arr.dtype}")

print("\nMetadata:")
print(json.dumps(task3_metadata, indent=2, ensure_ascii=False))

## 5. Preprocessing truth-level per sensitivity LLP

In [ ]:
TRUTH_SUMMARY_PATH = DATASET_DIR / "task3_signal_truth_summary.csv"


def extract_truth_from_processed_npz():
    candidates = [AGG_PATH, SEQ_PATH, TASK3_DATASET_PATH]
    rows = {}

    for p in candidates:
        npz = safe_load_npz(p)
        if npz is None:
            continue

        for key in npz.keys():
            kl = key.lower()
            arr = np.asarray(npz[key])

            if arr.ndim != 1:
                continue

            if any(tok in kl for tok in ["lxy", "ctau", "dark_pion", "truth", "mass"]):
                # Uso solo array plausibilmente allineati al segnale.
                if len(arr) == len(task3_data["agg_X_sig"]):
                    rows[key] = arr

    if not rows:
        return None

    df = pd.DataFrame(rows)
    return df


def summarize_truth_dark_pions_from_h5(h5_path, max_events=None):
    if h5py is None:
        print("[SKIP] h5py non disponibile.")
        return None

    h5_path = Path(h5_path)
    if not h5_path.exists():
        print("[SKIP] RAW_SIGNAL_H5_PATH non trovato:", h5_path)
        return None

    with h5py.File(h5_path, "r") as f:
        if "truth_dark_pions" not in f:
            print("[SKIP] truth_dark_pions non trovato in", h5_path)
            return None

        truth = f["truth_dark_pions"]
        n = len(truth) if max_events is None else min(len(truth), max_events)
        truth_arr = truth[:n]

    names = truth_arr.dtype.names
    if names is None:
        print("[SKIP] truth_dark_pions non è structured array.")
        return None

    valid = truth_arr["valid"].astype(bool) if "valid" in names else np.ones(truth_arr.shape[:2], dtype=bool)

    out = {
        "n_truth_dark_pions": valid.sum(axis=1).astype(np.float32),
    }

    def summarise_field(field):
        if field not in names:
            return

        values = np.asarray(truth_arr[field], dtype=np.float32)
        values = np.where(valid, values, np.nan)

        out[f"{field}_mean"] = np.nanmean(values, axis=1)
        out[f"{field}_median"] = np.nanmedian(values, axis=1)
        out[f"{field}_q90"] = np.nanquantile(values, 0.90, axis=1)
        out[f"{field}_max"] = np.nanmax(values, axis=1)

    for field in ["Lxy", "Lxy_prod", "mass", "pt", "energy"]:
        summarise_field(field)

    df = pd.DataFrame(out)
    df = df.replace([np.inf, -np.inf], np.nan)

    return df


truth_df = extract_truth_from_processed_npz()
truth_source = "processed_npz"

if truth_df is None and RAW_SIGNAL_H5_PATH is not None:
    truth_df = summarize_truth_dark_pions_from_h5(
        RAW_SIGNAL_H5_PATH,
        max_events=len(task3_data["agg_X_sig"]),
    )
    truth_source = "raw_signal_h5"

if truth_df is not None:
    # Allineamento prudente: taglio alla lunghezza del signal usato in Task 3.
    n_sig = len(task3_data["agg_X_sig"])
    truth_df = truth_df.iloc[:n_sig].reset_index(drop=True)
    truth_df["signal_index"] = np.arange(len(truth_df))
    truth_df.to_csv(TRUTH_SUMMARY_PATH, index=False)
    print("[OK] Truth summary salvata:", TRUTH_SUMMARY_PATH)
    print("Truth source:", truth_source)
    display(truth_df.head())
else:
    print("[INFO] Nessuna truth-level summary disponibile.")
    print("Per abilitarla, salva variabili truth nel preprocessing oppure imposta RAW_SIGNAL_H5_PATH.")

## 6. Caricamento score dai notebook dei modelli

In [ ]:
MODEL_SPECS = [
    {
        "family": "AE/VAE",
        "model": "AE/VAE",
        "dir": OUTPUTS_DIR / "ae_vae",
        "scores_npz": "ae_vae_scores.npz",
        "results_csv": "ae_vae_results.csv",
        "background_csv": None,
        "optional": False,
    },
    {
        "family": "Normalizing Flow",
        "model": "RealNVP Flow",
        "dir": OUTPUTS_DIR / "normalizing_flow",
        "scores_npz": "realnvp_scores.npz",
        "results_csv": "realnvp_results.csv",
        "background_csv": "realnvp_background_checks.csv",
        "optional": False,
    },
    {
        "family": "Diffusion",
        "model": "Diffusion",
        "dir": OUTPUTS_DIR / "diffusion",
        "scores_npz": "diffusion_scores.npz",
        "results_csv": "diffusion_results.csv",
        "background_csv": "diffusion_background_checks.csv",
        "optional": False,
    },
    {
        "family": "Transformer",
        "model": "Transformer Deep SVDD",
        "dir": OUTPUTS_DIR / "transformer_deep_svdd",
        "scores_npz": "transformer_deep_svdd_scores.npz",
        "results_csv": "transformer_deep_svdd_results.csv",
        "background_csv": "transformer_deep_svdd_background_checks.csv",
        "optional": False,
    },
    {
        "family": "GNN",
        "model": "GNN EdgeConv AE",
        "dir": OUTPUTS_DIR / "gnn_edgeconv_ae",
        "scores_npz": "gnn_edgeconv_ae_scores.npz",
        "results_csv": "gnn_edgeconv_ae_results.csv",
        "background_csv": "gnn_edgeconv_ae_background_checks.csv",
        "optional": True,
    },
    {
        "family": "GNN light",
        "model": "GNN light",
        "dir": OUTPUTS_DIR / "gnn_light",
        "scores_npz": None,
        "results_csv": "gnn_light_results.csv",
        "background_csv": "gnn_light_background_checks.csv",
        "optional": True,
    },
]


def classify_score_key(key):
    k = key.lower()

    if not any(tok in k for tok in ["score", "nll", "loss", "svdd"]):
        return None

    if any(tok in k for tok in ["fpr", "tpr", "threshold", "auc"]):
        return None

    if any(tok in k for tok in ["sig", "signal", "ej"]):
        split = "sig"
    elif any(tok in k for tok in ["bkg_test", "bg_test", "qcd_test", "test_bkg"]):
        split = "bkg_test"
    elif any(tok in k for tok in ["bkg_eval", "bg_eval", "qcd_eval", "eval_bkg"]):
        split = "bkg_eval"
    elif any(tok in k for tok in ["bkg_val", "bg_val", "qcd_val", "val_bkg"]):
        split = "bkg_val"
    elif any(tok in k for tok in ["bkg_train", "bg_train", "qcd_train", "train_bkg"]):
        split = "bkg_train"
    elif any(tok in k for tok in ["bkg", "bg", "qcd"]):
        split = "bkg"
    else:
        return None

    name = k
    replacements = [
        "scores", "score",
        "bkg_test", "bg_test", "qcd_test", "test_bkg",
        "bkg_eval", "bg_eval", "qcd_eval", "eval_bkg",
        "bkg_val", "bg_val", "qcd_val", "val_bkg",
        "bkg_train", "bg_train", "qcd_train", "train_bkg",
        "bkg", "bg", "qcd",
        "sig", "signal", "ej",
    ]

    for r in replacements:
        name = name.replace(r, "")

    name = name.strip("_- ")
    if not name:
        name = "default"

    return split, name


def collect_score_arrays(npz):
    collected = {}

    for key in npz.keys():
        arr = np.asarray(npz[key])

        if arr.ndim != 1:
            continue

        arr = arr.astype(np.float64)
        if not np.all(np.isfinite(arr)):
            arr = arr[np.isfinite(arr)]

        info = classify_score_key(key)
        if info is None:
            continue

        split, score_name = info
        collected.setdefault(score_name, {})
        collected[score_name][split] = arr

    return collected


def load_model_score_records():
    records = []

    for spec in MODEL_SPECS:
        model_dir = spec["dir"]

        if spec["scores_npz"] is None:
            npz_files = sorted(model_dir.glob("*scores*.npz")) if model_dir.exists() else []
        else:
            npz_files = [model_dir / spec["scores_npz"]]

        found_any = False

        for score_path in npz_files:
            npz = safe_load_npz(score_path)
            if npz is None:
                continue

            found_any = True
            collected = collect_score_arrays(npz)

            for score_name, splits in collected.items():
                sig = splits.get("sig")
                bg = (
                    splits.get("bkg_test")
                    if splits.get("bkg_test") is not None
                    else splits.get("bkg_eval")
                    if splits.get("bkg_eval") is not None
                    else splits.get("bkg")
                )

                if sig is None or bg is None:
                    continue

                records.append({
                    "family": spec["family"],
                    "model": spec["model"],
                    "score": score_name,
                    "score_path": score_path,
                    "bg": np.asarray(bg, dtype=np.float64),
                    "sig": np.asarray(sig, dtype=np.float64),
                    "train": splits.get("bkg_train"),
                    "val": splits.get("bkg_val") if splits.get("bkg_val") is not None else splits.get("bkg_eval"),
                    "test": splits.get("bkg_test"),
                })

        if not found_any:
            level = "[INFO]" if spec.get("optional", False) else "[WARN]"
            print(f"{level} score NPZ non trovato per {spec['model']} in {model_dir}")

    return records


score_records = load_model_score_records()

print(f"Score records caricati: {len(score_records)}")
for r in score_records:
    print(
        f"- {r['model']:28s} | {r['score']:25s} | "
        f"bg={len(r['bg']):7d} sig={len(r['sig']):7d} | {r['score_path']}"
    )

## 7. Metriche ROC/AUC, efficienze e bootstrap opzionale

In [ ]:
def orient_scores(bg, sig):
    bg = finite_1d(bg)
    sig = finite_1d(sig)

    y = np.concatenate([
        np.zeros(len(bg), dtype=np.int64),
        np.ones(len(sig), dtype=np.int64),
    ])

    s = np.concatenate([bg, sig])

    auc = roc_auc_score(y, s)
    auc_inv = roc_auc_score(y, -s)

    if auc_inv > auc:
        return -bg, -sig, auc_inv, True, auc, auc_inv

    return bg, sig, auc, False, auc, auc_inv


def evaluate_score(bg, sig, mistag_points=MISTAG_POINTS):
    bg_o, sig_o, best_auc, flipped, raw_auc, auc_inverted = orient_scores(bg, sig)

    y = np.concatenate([
        np.zeros(len(bg_o), dtype=np.int64),
        np.ones(len(sig_o), dtype=np.int64),
    ])

    s = np.concatenate([bg_o, sig_o])

    fpr, tpr, thresholds = roc_curve(y, s)

    eff_rows = []

    for mistag in mistag_points:
        threshold = np.quantile(bg_o, 1.0 - mistag)
        eff = float(np.mean(sig_o >= threshold))
        actual = float(np.mean(bg_o >= threshold))

        eff_rows.append({
            "mistag_target": float(mistag),
            "actual_mistag": actual,
            "signal_efficiency": eff,
            "threshold": float(threshold),
        })

    return {
        "bg_oriented": bg_o,
        "sig_oriented": sig_o,
        "best_auc": float(best_auc),
        "raw_auc": float(raw_auc),
        "auc_inverted": float(auc_inverted),
        "flipped": bool(flipped),
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "eff_table": pd.DataFrame(eff_rows),
    }


def bootstrap_score_ci(bg, sig, mistag_points=MISTAG_POINTS, n_boot=N_BOOTSTRAP):
    bg = np.asarray(bg)
    sig = np.asarray(sig)

    if len(bg) > BOOTSTRAP_MAX_EVENTS_PER_CLASS:
        idx = rng.choice(len(bg), size=BOOTSTRAP_MAX_EVENTS_PER_CLASS, replace=False)
        bg = bg[idx]

    if len(sig) > BOOTSTRAP_MAX_EVENTS_PER_CLASS:
        idx = rng.choice(len(sig), size=BOOTSTRAP_MAX_EVENTS_PER_CLASS, replace=False)
        sig = sig[idx]

    aucs = []
    effs = {m: [] for m in mistag_points}

    for _ in range(n_boot):
        ib = rng.integers(0, len(bg), size=len(bg))
        isg = rng.integers(0, len(sig), size=len(sig))

        ev = evaluate_score(bg[ib], sig[isg], mistag_points=mistag_points)
        aucs.append(ev["best_auc"])

        for _, row in ev["eff_table"].iterrows():
            effs[float(row["mistag_target"])].append(float(row["signal_efficiency"]))

    out = {
        "auc_low": np.percentile(aucs, 16),
        "auc_high": np.percentile(aucs, 84),
    }

    for m in mistag_points:
        out[f"eff_at_{m:g}_mistag_low"] = np.percentile(effs[m], 16)
        out[f"eff_at_{m:g}_mistag_high"] = np.percentile(effs[m], 84)

    return out


all_metric_rows = []
all_roc_rows = []
score_store = {}

for rec in score_records:
    ev = evaluate_score(rec["bg"], rec["sig"])

    key = (rec["model"], rec["score"])
    score_store[key] = {
        **rec,
        "eval": ev,
    }

    eff_df = ev["eff_table"].copy()
    eff_df["family"] = rec["family"]
    eff_df["model"] = rec["model"]
    eff_df["score"] = rec["score"]
    eff_df["auc"] = ev["best_auc"]
    eff_df["raw_auc"] = ev["raw_auc"]
    eff_df["auc_inverted"] = ev["auc_inverted"]
    eff_df["flipped"] = ev["flipped"]
    eff_df["n_bkg"] = len(rec["bg"])
    eff_df["n_sig"] = len(rec["sig"])
    eff_df["score_path"] = str(rec["score_path"])

    if DO_BOOTSTRAP_CI:
        ci = bootstrap_score_ci(rec["bg"], rec["sig"])
        eff_df["auc_low"] = ci["auc_low"]
        eff_df["auc_high"] = ci["auc_high"]

        for m in MISTAG_POINTS:
            mask_m = np.isclose(eff_df["mistag_target"], m)
            eff_df.loc[mask_m, "signal_efficiency_low"] = ci[f"eff_at_{m:g}_mistag_low"]
            eff_df.loc[mask_m, "signal_efficiency_high"] = ci[f"eff_at_{m:g}_mistag_high"]

    all_metric_rows.append(eff_df)

    for f, t in zip(ev["fpr"], ev["tpr"]):
        all_roc_rows.append({
            "family": rec["family"],
            "model": rec["model"],
            "score": rec["score"],
            "fpr": float(f),
            "tpr": float(t),
            "auc": ev["best_auc"],
            "flipped": ev["flipped"],
        })

metrics_long = pd.concat(all_metric_rows, ignore_index=True) if all_metric_rows else pd.DataFrame()
roc_curves = pd.DataFrame(all_roc_rows)

metrics_long.to_csv(TABLES_DIR / "task3_metrics_long.csv", index=False)
roc_curves.to_csv(TABLES_DIR / "task3_roc_curves.csv", index=False)

print("Saved:")
print(TABLES_DIR / "task3_metrics_long.csv")
print(TABLES_DIR / "task3_roc_curves.csv")

display(metrics_long.head())

## 8. Tabella riassuntiva e scelta score per modello

In [ ]:
# Score preferiti per report finale.
# Se non esistono, il notebook usa automaticamente lo score con AUC maggiore.
PREFERRED_SCORE_PATTERNS = {
    "AE/VAE": ["whitened", "official"],
    "RealNVP Flow": ["best", "nll", "score"],
    "Diffusion": ["z_all", "raw_all", "best"],
    "Transformer Deep SVDD": ["svdd", "score"],
    "GNN EdgeConv AE": ["total", "track"],
    "GNN light": ["total", "track"],
}


def build_summary(metrics_long):
    if metrics_long.empty:
        return pd.DataFrame()

    rows = []

    for (family, model, score), g in metrics_long.groupby(["family", "model", "score"], dropna=False):
        row = {
            "family": family,
            "model": model,
            "score": score,
            "auc": float(pd.to_numeric(g["auc"], errors="coerce").max()),
            "flipped": bool(g["flipped"].iloc[0]),
            "n_bkg": int(g["n_bkg"].iloc[0]),
            "n_sig": int(g["n_sig"].iloc[0]),
        }

        if "auc_low" in g.columns:
            row["auc_low"] = float(pd.to_numeric(g["auc_low"], errors="coerce").max())
            row["auc_high"] = float(pd.to_numeric(g["auc_high"], errors="coerce").max())

        for mistag in MISTAG_POINTS:
            gg = g[np.isclose(pd.to_numeric(g["mistag_target"], errors="coerce"), mistag)]
            if len(gg):
                row[f"eff_at_{mistag:g}_mistag"] = float(gg["signal_efficiency"].iloc[0])
                row[f"threshold_at_{mistag:g}_mistag"] = float(gg["threshold"].iloc[0])

                if "signal_efficiency_low" in gg.columns:
                    row[f"eff_at_{mistag:g}_mistag_low"] = float(gg["signal_efficiency_low"].iloc[0])
                    row[f"eff_at_{mistag:g}_mistag_high"] = float(gg["signal_efficiency_high"].iloc[0])

        rows.append(row)

    return pd.DataFrame(rows).sort_values("auc", ascending=False).reset_index(drop=True)


def choose_primary_scores(summary):
    if summary.empty:
        return summary

    selected = []

    for model, g in summary.groupby("model"):
        patterns = [p.lower() for p in PREFERRED_SCORE_PATTERNS.get(model, [])]
        chosen = None

        for pattern in patterns:
            gg = g[g["score"].astype(str).str.lower().str.contains(pattern, regex=False)]
            if len(gg):
                chosen = gg.sort_values("auc", ascending=False).iloc[0]
                break

        if chosen is None:
            chosen = g.sort_values("auc", ascending=False).iloc[0]

        selected.append(chosen)

    selected = pd.DataFrame(selected).sort_values("auc", ascending=False).reset_index(drop=True)
    return selected


summary_all_scores = build_summary(metrics_long)
summary_primary = choose_primary_scores(summary_all_scores)

summary_all_scores.to_csv(TABLES_DIR / "task3_summary_all_scores.csv", index=False)
summary_primary.to_csv(TABLES_DIR / "task3_summary_primary_scores.csv", index=False)

print("Saved:")
print(TABLES_DIR / "task3_summary_all_scores.csv")
print(TABLES_DIR / "task3_summary_primary_scores.csv")

display(summary_all_scores)
display(summary_primary)

## 9. Plot ROC, AUC ed efficienze a mistag fissato

In [ ]:
if not roc_curves.empty:
    plt.figure(figsize=(7, 7))

    for (model, score), g in roc_curves.groupby(["model", "score"]):
        # Per leggibilità mostro solo gli score selezionati nella tabella primaria.
        mask_primary = (
            (summary_primary["model"].astype(str) == str(model))
            & (summary_primary["score"].astype(str) == str(score))
        )

        if not mask_primary.any():
            continue

        g = g.sort_values("fpr")
        auc_value = float(g["auc"].iloc[0])
        plt.plot(g["fpr"], g["tpr"], label=f"{model} ({score}) AUC={auc_value:.4f}")

    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlabel("QCD mistag rate")
    plt.ylabel("Emerging Jets efficiency")
    plt.title("Task 3 ROC curves")
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "task3_roc_primary_scores.png", dpi=200)
    plt.show()


if not summary_primary.empty:
    plt.figure(figsize=(10, max(4, 0.45 * len(summary_primary))))
    plot_df = summary_primary.sort_values("auc", ascending=True)
    plt.barh(plot_df["model"] + " — " + plot_df["score"].astype(str), plot_df["auc"])
    plt.xlabel("AUC")
    plt.title("Task 3 AUC comparison")
    plt.xlim(0.0, 1.0)
    plt.grid(True, axis="x", alpha=0.35)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "task3_auc_primary_scores.png", dpi=200)
    plt.show()


for mistag in [0.10, 0.01, 0.001]:
    col = f"eff_at_{mistag:g}_mistag"
    if col not in summary_primary.columns:
        continue

    plt.figure(figsize=(10, max(4, 0.45 * len(summary_primary))))
    plot_df = summary_primary.sort_values(col, ascending=True)
    plt.barh(plot_df["model"] + " — " + plot_df["score"].astype(str), plot_df[col])
    plt.xlabel(f"EJ efficiency at {mistag:g} QCD mistag")
    plt.title(f"Task 3 efficiency at fixed QCD mistag = {mistag:g}")
    plt.xlim(0.0, 1.0)
    plt.grid(True, axis="x", alpha=0.35)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"task3_eff_at_{mistag:g}_mistag.png", dpi=200)
    plt.show()

## 10. Score distributions

In [ ]:
for _, row in summary_primary.iterrows():
    model = row["model"]
    score = row["score"]

    key = (model, score)

    if key not in score_store:
        continue

    ev = score_store[key]["eval"]
    bg = ev["bg_oriented"]
    sig = ev["sig_oriented"]

    all_scores = np.concatenate([bg, sig])
    lo = np.percentile(all_scores, 0.1)
    hi = np.percentile(all_scores, 99.5)

    if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
        lo, hi = np.min(all_scores), np.max(all_scores)

    plt.figure(figsize=(8, 5))
    plt.hist(bg, bins=100, range=(lo, hi), density=True, alpha=0.6, label="QCD test")
    plt.hist(sig, bins=100, range=(lo, hi), density=True, alpha=0.6, label="EJ signal")
    plt.yscale("log")
    plt.xlabel("oriented anomaly score")
    plt.ylabel("density")
    plt.title(f"Score distribution — {model} ({score})")
    plt.legend()
    plt.grid(True, alpha=0.35)
    plt.tight_layout()

    safe_name = (
        str(model).lower().replace(" ", "_").replace("/", "_")
        + "__"
        + str(score).lower().replace(" ", "_").replace("/", "_")
    )

    plt.savefig(PLOTS_DIR / f"score_distribution_{safe_name}.png", dpi=200)
    plt.show()

## 11. Consistency checks: QCD train / validation / test

In [ ]:
def bg_vs_bg_auc(a, b, name_a, name_b):
    if a is None or b is None:
        return None

    a = finite_1d(a)
    b = finite_1d(b)

    if len(a) == 0 or len(b) == 0:
        return None

    y = np.concatenate([
        np.zeros(len(a), dtype=np.int64),
        np.ones(len(b), dtype=np.int64),
    ])

    s = np.concatenate([a, b])

    auc = roc_auc_score(y, s)
    auc_inv = roc_auc_score(y, -s)

    return {
        "comparison": f"{name_a} vs {name_b}",
        "auc": float(auc),
        "auc_inverted": float(auc_inv),
        "best_auc": float(max(auc, auc_inv)),
    }


bg_check_rows = []

for rec in score_records:
    comparisons = [
        ("train QCD", rec.get("train"), "val QCD", rec.get("val")),
        ("train QCD", rec.get("train"), "test QCD", rec.get("test")),
        ("val QCD", rec.get("val"), "test QCD", rec.get("test")),
    ]

    for name_a, a, name_b, b in comparisons:
        out = bg_vs_bg_auc(a, b, name_a, name_b)

        if out is None:
            continue

        out["family"] = rec["family"]
        out["model"] = rec["model"]
        out["score"] = rec["score"]
        bg_check_rows.append(out)

background_checks = pd.DataFrame(bg_check_rows)
background_checks.to_csv(TABLES_DIR / "background_consistency_checks.csv", index=False)

print("Saved:", TABLES_DIR / "background_consistency_checks.csv")
display(background_checks)

if not background_checks.empty:
    plt.figure(figsize=(10, max(4, 0.45 * len(background_checks))))
    labels = (
        background_checks["model"].astype(str)
        + " — "
        + background_checks["score"].astype(str)
        + " — "
        + background_checks["comparison"].astype(str)
    )
    order = np.argsort(background_checks["best_auc"].values)
    plt.barh(labels.iloc[order], background_checks["best_auc"].iloc[order])
    plt.axvline(0.5, linestyle="--", linewidth=1, label="ideal consistency")
    plt.xlabel("Best AUC separating two QCD samples")
    plt.title("QCD-vs-QCD consistency checks")
    plt.xlim(0.45, 1.0)
    plt.grid(True, axis="x", alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "background_consistency_checks.png", dpi=200)
    plt.show()

## 12. Supervised upper bound

In [ ]:
# Questa sezione usa le label QCD/EJ e quindi NON è anomaly detection.
# Serve come upper bound supervisionato per capire quanta informazione discriminante
# è contenuta nelle feature aggregate.

X_bkg_train = np.asarray(task3_data["agg_X_bkg_train"], dtype=np.float32)
X_bkg_test = np.asarray(task3_data["agg_X_bkg_test"], dtype=np.float32)
X_sig_all = np.asarray(task3_data["agg_X_sig"], dtype=np.float32)
feature_names = as_list_of_str(task3_data["agg_feature_names"])

check_no_forbidden_features(feature_names, "supervised aggregate features")

idx_sig_train, idx_sig_test = train_test_split(
    np.arange(len(X_sig_all)),
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
)

X_sig_train = X_sig_all[idx_sig_train]
X_sig_test = X_sig_all[idx_sig_test]

def balanced_sample(X, n, seed):
    if len(X) <= n:
        return X
    rr = np.random.default_rng(seed)
    idx = rr.choice(len(X), size=n, replace=False)
    return X[idx]

n_train_per_class = min(
    len(X_bkg_train),
    len(X_sig_train),
    SUPERVISED_MAX_TRAIN_PER_CLASS,
)

X_bkg_train_sup = balanced_sample(X_bkg_train, n_train_per_class, SEED)
X_sig_train_sup = balanced_sample(X_sig_train, n_train_per_class, SEED + 1)

if SUPERVISED_MAX_TEST_PER_CLASS is None:
    n_test_bkg = len(X_bkg_test)
    n_test_sig = len(X_sig_test)
else:
    n_test_bkg = min(len(X_bkg_test), SUPERVISED_MAX_TEST_PER_CLASS)
    n_test_sig = min(len(X_sig_test), SUPERVISED_MAX_TEST_PER_CLASS)

X_bkg_test_sup = balanced_sample(X_bkg_test, n_test_bkg, SEED + 2)
X_sig_test_sup = balanced_sample(X_sig_test, n_test_sig, SEED + 3)

X_sup_train = np.concatenate([X_bkg_train_sup, X_sig_train_sup], axis=0)
y_sup_train = np.concatenate([
    np.zeros(len(X_bkg_train_sup), dtype=np.int64),
    np.ones(len(X_sig_train_sup), dtype=np.int64),
])

X_sup_test = np.concatenate([X_bkg_test_sup, X_sig_test_sup], axis=0)
y_sup_test = np.concatenate([
    np.zeros(len(X_bkg_test_sup), dtype=np.int64),
    np.ones(len(X_sig_test_sup), dtype=np.int64),
])

print("Supervised train:", X_sup_train.shape, np.bincount(y_sup_train))
print("Supervised test:", X_sup_test.shape, np.bincount(y_sup_test))

supervised_models = {
    "Supervised Logistic": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, class_weight="balanced"),
    ),
    "Supervised GBT": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        l2_regularization=1e-4,
        random_state=SEED,
    ),
}

supervised_rows = []
supervised_roc_rows = []

for name, clf in supervised_models.items():
    clf.fit(X_sup_train, y_sup_train)

    if hasattr(clf, "predict_proba"):
        score = clf.predict_proba(X_sup_test)[:, 1]
    else:
        score = clf.decision_function(X_sup_test)

    bg_score = score[y_sup_test == 0]
    sig_score = score[y_sup_test == 1]

    ev = evaluate_score(bg_score, sig_score)

    for _, r in ev["eff_table"].iterrows():
        row = {
            "family": "Supervised upper bound",
            "model": name,
            "score": "p(EJ)",
            "auc": ev["best_auc"],
            "mistag_target": r["mistag_target"],
            "actual_mistag": r["actual_mistag"],
            "signal_efficiency": r["signal_efficiency"],
            "threshold": r["threshold"],
            "n_bkg": len(bg_score),
            "n_sig": len(sig_score),
        }
        supervised_rows.append(row)

    for f, t in zip(ev["fpr"], ev["tpr"]):
        supervised_roc_rows.append({
            "family": "Supervised upper bound",
            "model": name,
            "score": "p(EJ)",
            "fpr": float(f),
            "tpr": float(t),
            "auc": ev["best_auc"],
        })

supervised_metrics = pd.DataFrame(supervised_rows)
supervised_roc = pd.DataFrame(supervised_roc_rows)

supervised_metrics.to_csv(TABLES_DIR / "supervised_upper_bound_metrics.csv", index=False)
supervised_roc.to_csv(TABLES_DIR / "supervised_upper_bound_roc.csv", index=False)

print("Saved:")
print(TABLES_DIR / "supervised_upper_bound_metrics.csv")
print(TABLES_DIR / "supervised_upper_bound_roc.csv")

display(supervised_metrics.head())

## 13. Supervised feature importance

In [ ]:
# Coefficienti della Logistic Regression.
logistic_model = supervised_models.get("Supervised Logistic")

if logistic_model is not None:
    clf = logistic_model.named_steps["logisticregression"]
    scaler = logistic_model.named_steps["standardscaler"]

    coef = clf.coef_.reshape(-1)

    coef_df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coef,
        "abs_coefficient": np.abs(coef),
    }).sort_values("abs_coefficient", ascending=False)

    coef_df.to_csv(TABLES_DIR / "supervised_logistic_feature_coefficients.csv", index=False)
    display(coef_df.head(30))

    top = coef_df.head(20).iloc[::-1]

    plt.figure(figsize=(8, 7))
    plt.barh(top["feature"], top["coefficient"])
    plt.xlabel("standardized logistic coefficient")
    plt.title("Supervised Logistic feature coefficients")
    plt.grid(True, axis="x", alpha=0.35)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "supervised_logistic_feature_coefficients.png", dpi=200)
    plt.show()


# Permutation importance per GBT su subset.
gbt_model = supervised_models.get("Supervised GBT")

if gbt_model is not None:
    max_perm = min(20_000, len(X_sup_test))
    idx = rng.choice(len(X_sup_test), size=max_perm, replace=False)

    perm = permutation_importance(
        gbt_model,
        X_sup_test[idx],
        y_sup_test[idx],
        n_repeats=5,
        random_state=SEED,
        scoring="roc_auc",
    )

    perm_df = pd.DataFrame({
        "feature": feature_names,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }).sort_values("importance_mean", ascending=False)

    perm_df.to_csv(TABLES_DIR / "supervised_gbt_permutation_importance.csv", index=False)
    display(perm_df.head(30))

    top = perm_df.head(20).iloc[::-1]

    plt.figure(figsize=(8, 7))
    plt.barh(top["feature"], top["importance_mean"])
    plt.xlabel("permutation importance in ROC AUC")
    plt.title("Supervised GBT feature importance")
    plt.grid(True, axis="x", alpha=0.35)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "supervised_gbt_permutation_importance.png", dpi=200)
    plt.show()

## 14. Sensitivity vs truth-level LLP variables

In [ ]:
def make_truth_bins(x, n_bins=N_TRUTH_BINS):
    x = np.asarray(x, dtype=np.float64)
    valid = np.isfinite(x)

    if valid.sum() < max(20, n_bins):
        return None, None

    xv = x[valid]

    if np.nanmax(xv) - np.nanmin(xv) <= 1e-12:
        return None, None

    edges = np.unique(np.nanquantile(xv, np.linspace(0, 1, n_bins + 1)))

    if len(edges) < 3:
        return None, None

    return edges, valid


def efficiency_vs_truth_variable(sig_scores, bg_scores, truth_values, mistag):
    sig_scores = np.asarray(sig_scores, dtype=np.float64)
    bg_scores = np.asarray(bg_scores, dtype=np.float64)
    truth_values = np.asarray(truth_values, dtype=np.float64)

    n = min(len(sig_scores), len(truth_values))
    sig_scores = sig_scores[:n]
    truth_values = truth_values[:n]

    threshold = np.quantile(bg_scores, 1.0 - mistag)

    edges, valid = make_truth_bins(truth_values)

    if edges is None:
        return pd.DataFrame()

    rows = []

    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = valid & (truth_values >= lo) & (truth_values <= hi)

        if in_bin.sum() == 0:
            continue

        eff = float(np.mean(sig_scores[in_bin] >= threshold))

        rows.append({
            "bin_low": float(lo),
            "bin_high": float(hi),
            "bin_center": float(0.5 * (lo + hi)),
            "n_signal": int(in_bin.sum()),
            "signal_efficiency": eff,
            "threshold": float(threshold),
            "mistag_target": float(mistag),
        })

    return pd.DataFrame(rows)


truth_sensitivity_rows = []

if DO_TRUTH_SENSITIVITY and truth_df is not None and not summary_primary.empty:
    candidate_truth_columns = []

    for c in truth_df.columns:
        cl = c.lower()
        if c == "signal_index":
            continue
        if any(tok in cl for tok in ["lxy", "ctau", "mass", "n_truth_dark_pions"]):
            candidate_truth_columns.append(c)

    print("Truth columns considered:", candidate_truth_columns)

    for _, row in summary_primary.iterrows():
        key = (row["model"], row["score"])

        if key not in score_store:
            continue

        ev = score_store[key]["eval"]
        bg_scores = ev["bg_oriented"]
        sig_scores = ev["sig_oriented"]

        for var in candidate_truth_columns:
            values = truth_df[var].values

            for mistag in TRUTH_MISTAG_POINTS:
                df_var = efficiency_vs_truth_variable(
                    sig_scores=sig_scores,
                    bg_scores=bg_scores,
                    truth_values=values,
                    mistag=mistag,
                )

                if df_var.empty:
                    continue

                df_var["family"] = row["family"]
                df_var["model"] = row["model"]
                df_var["score"] = row["score"]
                df_var["truth_variable"] = var

                truth_sensitivity_rows.append(df_var)

truth_sensitivity = (
    pd.concat(truth_sensitivity_rows, ignore_index=True)
    if truth_sensitivity_rows
    else pd.DataFrame()
)

truth_sensitivity.to_csv(TABLES_DIR / "truth_level_sensitivity.csv", index=False)
print("Saved:", TABLES_DIR / "truth_level_sensitivity.csv")

display(truth_sensitivity.head())


if not truth_sensitivity.empty:
    for (model, score, mistag, var), g in truth_sensitivity.groupby(
        ["model", "score", "mistag_target", "truth_variable"]
    ):
        plt.figure(figsize=(7, 5))
        plt.plot(g["bin_center"], g["signal_efficiency"], marker="o")
        plt.xlabel(var)
        plt.ylabel(f"EJ efficiency at {mistag:g} QCD mistag")
        plt.title(f"Sensitivity vs {var}\n{model} ({score})")
        plt.grid(True, alpha=0.35)
        plt.tight_layout()

        safe = (
            str(model).lower().replace(" ", "_").replace("/", "_")
            + "__"
            + str(score).lower().replace(" ", "_").replace("/", "_")
            + "__"
            + str(var).lower().replace(" ", "_").replace("/", "_")
            + f"__mistag_{mistag:g}"
        )

        plt.savefig(PLOTS_DIR / f"truth_sensitivity_{safe}.png", dpi=200)
        plt.show()
else:
    print("[INFO] Nessun plot truth-level prodotto. Controlla che le truth variables siano disponibili e non costanti.")

## 15. Latent space PCA, se embedding disponibili

In [ ]:
def collect_embedding_keys(npz):
    out = {}

    for key in npz.keys():
        arr = np.asarray(npz[key])
        kl = key.lower()

        if arr.ndim != 2:
            continue

        if not any(tok in kl for tok in ["latent", "embedding", "z_"]):
            continue

        if any(tok in kl for tok in ["sig", "signal", "ej"]):
            split = "sig"
        elif any(tok in kl for tok in ["bkg_test", "bg_test", "qcd_test", "test"]):
            split = "bkg"
        elif any(tok in kl for tok in ["bkg", "bg", "qcd"]):
            split = "bkg"
        else:
            continue

        name = kl
        for r in ["latent", "embeddings", "embedding", "z_", "bkg_test", "bg_test", "qcd_test", "bkg", "bg", "qcd", "sig", "signal", "ej"]:
            name = name.replace(r, "")

        name = name.strip("_- ") or "latent"
        out.setdefault(name, {})
        out[name][split] = arr

    return out


MAX_PCA_POINTS_PER_CLASS = 5000

for spec in MODEL_SPECS:
    model_dir = spec["dir"]

    if spec["scores_npz"] is None:
        npz_files = sorted(model_dir.glob("*scores*.npz")) if model_dir.exists() else []
    else:
        npz_files = [model_dir / spec["scores_npz"]]

    for p in npz_files:
        npz = safe_load_npz(p)

        if npz is None:
            continue

        emb = collect_embedding_keys(npz)

        for name, splits in emb.items():
            if "bkg" not in splits or "sig" not in splits:
                continue

            Zb = np.asarray(splits["bkg"], dtype=np.float32)
            Zs = np.asarray(splits["sig"], dtype=np.float32)

            nb = min(MAX_PCA_POINTS_PER_CLASS, len(Zb))
            ns = min(MAX_PCA_POINTS_PER_CLASS, len(Zs))

            ib = rng.choice(len(Zb), size=nb, replace=False)
            isg = rng.choice(len(Zs), size=ns, replace=False)

            X = np.concatenate([Zb[ib], Zs[isg]], axis=0)
            y = np.concatenate([
                np.zeros(nb, dtype=np.int64),
                np.ones(ns, dtype=np.int64),
            ])

            Xs = StandardScaler().fit_transform(X)
            pca = PCA(n_components=2, random_state=SEED)
            X2 = pca.fit_transform(Xs)

            plt.figure(figsize=(7, 6))
            plt.scatter(X2[y == 0, 0], X2[y == 0, 1], s=6, alpha=0.45, label="QCD")
            plt.scatter(X2[y == 1, 0], X2[y == 1, 1], s=6, alpha=0.45, label="EJ")
            plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%})")
            plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%})")
            plt.title(f"Latent PCA — {spec['model']} ({name})")
            plt.legend()
            plt.grid(True, alpha=0.35)
            plt.tight_layout()

            safe = (
                str(spec["model"]).lower().replace(" ", "_").replace("/", "_")
                + "__"
                + str(name).lower().replace(" ", "_").replace("/", "_")
            )

            plt.savefig(PLOTS_DIR / f"latent_pca_{safe}.png", dpi=200)
            plt.show()

## 16. Final export: tabella unica per report

In [ ]:
# Combino unsupervised primary scores e supervised upper bound in una tabella unica.
report_rows = []

if not summary_primary.empty:
    report_rows.append(summary_primary.copy())

if not supervised_metrics.empty:
    supervised_summary = build_summary(supervised_metrics)
    report_rows.append(supervised_summary)

if report_rows:
    report_summary = pd.concat(report_rows, ignore_index=True, sort=False)
    report_summary = report_summary.sort_values("auc", ascending=False).reset_index(drop=True)
else:
    report_summary = pd.DataFrame()

report_summary.to_csv(TABLES_DIR / "task3_report_summary.csv", index=False)

display(report_summary)

print("Saved:", TABLES_DIR / "task3_report_summary.csv")
print("All Task 3 outputs written under:", TASK3_DIR.resolve())

## 17. Nota finale per la relazione

Questa cella non interpreta automaticamente i risultati, perché l'interpretazione va fatta dopo il run finale con la statistica scelta.

Punti che la relazione dovrà discutere dopo l'esecuzione:

1. confronto tra AUC integrata e efficienza di segnale a basso QCD mistag;
2. consistenza QCD train/validation/test;
3. eventuali differenze tra modelli tabulari e modelli track-set;
4. ruolo del supervised baseline come upper bound;
5. sensitivity in funzione delle variabili truth-level LLP, se disponibili;
6. limitazioni dovute alla statistica del campione e alla dimensione del run locale/HPC.

La tabella principale da usare nel report è:

```text
FINAL/outputs/task3_evaluation/tables/task3_report_summary.csv
```

Le figure principali sono in:

```text
FINAL/outputs/task3_evaluation/plots/
```